In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/cleaned/fraud_features.csv")
df.columns = df.columns.str.replace(" ", "_")

conn = sqlite3.connect("../data/cleaned/fraud.db")
df.to_sql("claims", conn, if_exists="replace", index=False)

print(df.shape)

(15419, 50)


In [2]:
def q(sql):
    return pd.read_sql(sql, conn)

q("SELECT COUNT(*) AS total_claims FROM claims")

,total_claims
0,15419


In [3]:
q("""
SELECT COUNT(*)                                       AS total_claims,
       SUM(FraudFound_P)                               AS fraud_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
""")

,total_claims,fraud_claims,fraud_rate_pct
0,15419,923,5.99


In [4]:
q("""
SELECT PolicyType,
       COUNT(*)                                       AS n_claims,
       SUM(FraudFound_P)                               AS n_fraud,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
GROUP BY PolicyType
HAVING COUNT(*) >= 50
ORDER BY fraud_rate_pct DESC
""")

,PolicyType,n_claims,n_fraud,fraud_rate_pct
0,Sport - Collision,348,48,13.79
1,Utility - All Perils,340,41,12.06
2,Sedan - All Perils,4086,411,10.06
3,Sedan - Collision,5584,384,6.88
4,Sedan - Liability,4987,36,0.72


In [5]:
q("""
SELECT Fault, PoliceReportFiled,
       COUNT(*)                                       AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
GROUP BY Fault, PoliceReportFiled
ORDER BY fraud_rate_pct DESC
""")

,Fault,PoliceReportFiled,n_claims,fraud_rate_pct
0,Policy Holder,No,10948,7.96
1,Policy Holder,Yes,281,5.34
2,Third Party,No,4043,0.89
3,Third Party,Yes,147,0.68


In [6]:
q("""
SELECT AddressChange_Claim,
       COUNT(*)                                       AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
GROUP BY AddressChange_Claim
ORDER BY fraud_rate_pct DESC
""")

,AddressChange_Claim,n_claims,fraud_rate_pct
0,under 6 months,4,75.00
1,2 to 3 years,291,17.53
2,1 year,170,6.47
3,no change,14323,5.76
4,4 to 8 years,631,5.23


In [7]:
q("""
SELECT PastNumberOfClaims,
       COUNT(*)                                       AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
GROUP BY PastNumberOfClaims
ORDER BY PastNumberOfClaims_ord
""")

,PastNumberOfClaims,n_claims,fraud_rate_pct
0,none,4351,7.79
1,1,3573,6.21
2,2 to 4,5485,5.36
3,more than 4,2010,3.38


In [8]:
q("""
SELECT PastNumberOfClaims, PastNumberOfClaims_ord,
       COUNT(*)                                       AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2)              AS fraud_rate_pct
FROM claims
GROUP BY PastNumberOfClaims, PastNumberOfClaims_ord
ORDER BY PastNumberOfClaims_ord
""")

,PastNumberOfClaims,PastNumberOfClaims_ord,n_claims,fraud_rate_pct
0,none,0,4351,7.79
1,1,1,3573,6.21
2,2 to 4,2,5485,5.36
3,more than 4,3,2010,3.38


In [9]:
q("""
WITH make_stats AS (
    SELECT Make,
           COUNT(*) AS n_claims,
           SUM(FraudFound_P) AS n_fraud,
           100.0 * AVG(FraudFound_P) AS fraud_rate_pct,
           RANK() OVER (ORDER BY AVG(FraudFound_P) DESC) AS fraud_rank
    FROM claims
    GROUP BY Make
    HAVING COUNT(*) >= 50
)
SELECT * FROM make_stats ORDER BY fraud_rank
""")

,Make,n_claims,n_fraud,fraud_rate_pct,fraud_rank
0,Accura,472,59,12.500000,1
1,Saturn,58,6,10.344828,2
2,Saab,108,11,10.185185,3
3,Ford,450,33,7.333333,4
4,Mercury,83,6,7.228916,5
5,Honda,2800,179,6.392857,6
6,Toyota,3121,186,5.959628,7
7,Chevrolet,1681,94,5.591910,8
8,Pontiac,3837,213,5.551212,9
9,Mazda,2354,123,5.225149,10


In [10]:
sql_content = """
-- Overall fraud snapshot
SELECT COUNT(*) AS total_claims, SUM(FraudFound_P) AS fraud_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2) AS fraud_rate_pct FROM claims;

-- H3: Fraud rate by policy type
SELECT PolicyType, COUNT(*) AS n_claims, SUM(FraudFound_P) AS n_fraud,
       ROUND(100.0 * AVG(FraudFound_P), 2) AS fraud_rate_pct
FROM claims GROUP BY PolicyType HAVING COUNT(*) >= 50 ORDER BY fraud_rate_pct DESC;

-- H4: Fraud rate by fault and police report
SELECT Fault, PoliceReportFiled, COUNT(*) AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2) AS fraud_rate_pct
FROM claims GROUP BY Fault, PoliceReportFiled ORDER BY fraud_rate_pct DESC;

-- Fraud rate by address change before claim
SELECT AddressChange_Claim, COUNT(*) AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2) AS fraud_rate_pct
FROM claims GROUP BY AddressChange_Claim ORDER BY fraud_rate_pct DESC;

-- H5: Fraud rate by past claims history
SELECT PastNumberOfClaims, PastNumberOfClaims_ord, COUNT(*) AS n_claims,
       ROUND(100.0 * AVG(FraudFound_P), 2) AS fraud_rate_pct
FROM claims GROUP BY PastNumberOfClaims, PastNumberOfClaims_ord ORDER BY PastNumberOfClaims_ord;

-- Fraud rank by vehicle make (window function)
WITH make_stats AS (
    SELECT Make, COUNT(*) AS n_claims, SUM(FraudFound_P) AS n_fraud,
           100.0 * AVG(FraudFound_P) AS fraud_rate_pct,
           RANK() OVER (ORDER BY AVG(FraudFound_P) DESC) AS fraud_rank
    FROM claims GROUP BY Make HAVING COUNT(*) >= 50
)
SELECT * FROM make_stats ORDER BY fraud_rank;
"""

with open("../sql/01_fraud_analysis.sql", "w") as f:
    f.write(sql_content)
print("Saved.")

Saved.


## SQL Findings Summary
- H3 (policy type): Confirmed. 19x spread — Sport-Collision (13.79%) vs Sedan-Liability (0.72%).
- H4 (fault/police report): Partially confirmed. Fault is the dominant factor (7.96% vs 0.89%);
  missing police report adds a smaller secondary effect within each fault group.
- Address change before claim: Directionally supports fraud risk, but the strongest bucket
  (under 6 months, 75%) has only 4 claims — too small to trust on its own.
- H5 (past claims): Rejected — fraud rate falls as claim history grows (7.79% → 3.38%),
  opposite of the hypothesis. Likely reflects that repeat customers are lower-risk, known entities.
- Operational insight: Accura, Saturn, Saab rank highest in fraud rate per claim filed —
  useful for prioritizing investigator review by vehicle make.